## **Laboratorio Clase 1: Explorando la API de un LLM**

Usaremos noticias tecnológicas como hilo conductor para explorar cómo funciona la API de OpenAI:
desde cómo el modelo ve el texto hasta técnicas de prompt engineering, con un modelo de razonamiento actual (**GPT-6 Luna**).

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **Token** | el trozo de texto con el que trabaja el modelo (una palabra corta o parte de una). Se paga por token. |
| **API** | la forma de usar el modelo desde código: envías una petición y recibes la respuesta. |
| **Responses API** | la API de OpenAI que usamos en el curso. Chat Completions es la anterior, que imitan otros proveedores. |
| **`instructions`** | el prompt de sistema: las reglas fijas que sigue el modelo en toda la conversación. |
| **`reasoning.effort`** | cuánto «piensa» el modelo antes de responder: más esfuerzo, mejor en lo difícil pero más lento y caro. |


### Qué practicarás

- Tokenización: cómo convierte el modelo el texto en números
- **Responses API vs Chat Completions**: la misma llamada con las dos APIs
- Parámetros de generación en modelos de razonamiento: `reasoning.effort`, `max_output_tokens`, `verbosity` (y por qué `temperature` ya no es el mando principal)
- Conversación multi-turno: historial manual vs memoria en el servidor (`previous_response_id`)
- Prompt engineering: system prompts, few-shot y razonamiento

### Objetivos

Al finalizar serás capaz de:

- Conectar con la API y hacer llamadas básicas
- Contar tokens y entender cómo afectan al coste
- Elegir el nivel de razonamiento adecuado para cada tarea (coste y latencia frente a calidad)
- Construir un chatbot que recuerda el contexto de la conversación
- Aplicar técnicas de prompt engineering a tareas NLP reales

## Sección 1: Configuración del entorno

**Celda 1: Instalación de librerías**

In [ ]:
!pip install -q --upgrade openai tiktoken

`openai`: librería oficial de OpenAI. `tiktoken`: el tokenizador de OpenAI, que nos permite ver cómo se convierte el texto en tokens numéricos. Ilustra que los modelos *no entienden palabras, entienden números*.

**Celda 2: Clave de API**

In [ ]:
import os, getpass

# Orden: secretos de Colab -> variable de entorno -> teclado (sin mostrarla en pantalla)
try:  # en Colab: la clave guardada en Secretos (icono de la llave) con el nombre OPENAI_API_KEY
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

La clave de API identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook ni la subas a un repositorio público. En Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; si no la encuentra, te la pide con `getpass`, sin mostrarla en pantalla.

**Celda 3: Cliente, modelo y función auxiliar**

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

MODELO = "gpt-6-luna"   # el modelo GPT-6 más rápido y barato
ESFUERZO = "low"         # nivel de razonamiento por defecto: none | low | medium | high | xhigh | max

def preguntar(pregunta, esfuerzo=ESFUERZO, max_tokens=2000):
    r = client.responses.create(
        model=MODELO,
        input=pregunta,
        reasoning={"effort": esfuerzo},
        max_output_tokens=max_tokens,
    )
    return r.output_text

print("Cliente listo. Probando...")
print(preguntar("Di hola en 5 palabras."))

Creamos `preguntar()` con el nivel de razonamiento y la longitud configurables. Centralizamos el modelo en `MODELO`: para cambiarlo en todo el notebook basta con tocar una línea.

> **Modelos de razonamiento:** GPT-6 «piensa» (genera tokens de razonamiento internos) antes de responder. Con `reasoning.effort` decides cuánto: `none` responde directamente; `low`, `medium`, `high`... piensan cada vez más (mejor en problemas difíciles, pero más lento y caro).

## Sección 2: Responses API vs Chat Completions

OpenAI tiene dos APIs para generar texto:

| | **Chat Completions** (2023) | **Responses** (2025, la recomendada) |
|---|---|---|
| Endpoint | `client.chat.completions.create` | `client.responses.create` |
| Entrada | lista `messages` | `input` (texto o lista de mensajes) + `instructions` |
| Salida | `choices[0].message.content` | `output_text` (o la lista `output` de items) |
| Memoria | la gestionas tú (reenvías el historial) | opcional en el servidor (`previous_response_id`) |
| Tools integradas (búsqueda web, ficheros...) | no | sí |
| Razonamiento + tools en GPT-6 | **no** (solo con `effort="none"`) | sí |

¿Por qué conocer las dos? Chat Completions es el formato «compatible con OpenAI» que copian casi todos los demás (Gemini, DeepSeek, Ollama, vLLM, LM Studio...) y el que verás en la mayoría de tutoriales. En este curso usaremos **Responses**.

**Celda 4: La misma petición con las dos APIs**

In [ ]:
pregunta = "Explica en una frase qué es un LLM."

# --- Chat Completions ---
cc = client.chat.completions.create(
    model=MODELO,
    reasoning_effort=ESFUERZO,
    messages=[
        {"role": "system", "content": "Eres un profesor de IA. Respondes de forma breve."},
        {"role": "user", "content": pregunta},
    ],
)
print("Chat Completions:", cc.choices[0].message.content)

# --- Responses ---
r = client.responses.create(
    model=MODELO,
    reasoning={"effort": ESFUERZO},
    instructions="Eres un profesor de IA. Respondes de forma breve.",
    input=pregunta,
)
print("Responses:       ", r.output_text)

Las dos dan el mismo resultado. Fíjate en las diferencias de forma: el system prompt pasa a `instructions`, y la respuesta se lee con `output_text` en lugar de `choices[0].message.content`.

**Celda 5: ¿Qué hay dentro de una respuesta?**

In [ ]:
for item in r.output:
    print(item.type)

u = r.usage
print()
print(f"Tokens de entrada:       {u.input_tokens}")
print(f"Tokens de salida:        {u.output_tokens}")
print(f"  ...de razonamiento:    {u.output_tokens_details.reasoning_tokens}  (no los ves, pero se pagan)")

`output` es una lista de *items*: aparece un item `reasoning` (el modelo pensó) y un `message` (la respuesta). Los tokens de razonamiento se cuentan y se cobran como tokens de salida.

## Sección 3: Tokenización: cómo ve el modelo el texto

Las diapositivas mostraron que los algoritmos no entienden palabras, entienden números. Aquí lo verificamos directamente.

**Celda 6: Contar tokens en distintos textos**

In [ ]:
import tiktoken

# tiktoken todavía no reconoce gpt-6-luna; o200k_base es el tokenizador de la familia GPT-4o/GPT-5 y sirve como aproximación
enc = tiktoken.get_encoding("o200k_base")

textos = [
    "Hola",
    "inteligencia artificial",
    "Los LLMs no entienden palabras, entienden tokens.",
    "The quick brown fox jumps over the lazy dog.",
]

print(f"{'Texto':<52} {'Tokens':>6}  IDs")
print("-" * 80)
for texto in textos:
    ids = enc.encode(texto)
    ids_str = str(ids[:5]) + "..." if len(ids) > 5 else str(ids)
    print(f"{texto[:50]:<52} {len(ids):>6}  {ids_str}")

**Observaciones clave:**
- El español suele usar más tokens que el inglés para el mismo significado (mayor coste)
- Los tokens son subpalabras, no palabras completas
- Lo que el modelo «ve» son estos números, no las letras

**Celda 7: Decodificar: ver las «piezas» de cada token**

In [ ]:
frase = "inteligencia artificial"
tokens = enc.encode(frase)

print(f'Frase: "{frase}"')
print(f"IDs numéricos: {tokens}")
print()
print("Cada token decodificado:")
for t in tokens:
    print(f'  ID {t:6d}  ->  "{enc.decode([t])}"')

print()
print("--- Comparación español vs inglés (mismo significado) ---")
pares = [("casa", "house"), ("electroencefalografista", "electroencephalographer"), ("modelo", "model")]
for es, en in pares:
    print(f"{es:25s} {len(enc.encode(es))} tok  |  {en:25s} {len(enc.encode(en))} tok")

Esto explica por qué los prompts en español cuestan más que en inglés: más tokens = más coste.

**Celda 8: El recuento exacto, preguntando al propio modelo**

In [ ]:
texto = "Los LLMs no entienden palabras, entienden tokens."

exacto = client.responses.input_tokens.count(model=MODELO, input=texto)
print(f"tiktoken (aproximado): {len(enc.encode(texto))} tokens")
print(f"API (exacto):          {exacto.input_tokens} tokens")

`tiktoken` funciona sin conexión y es gratis, pero es una aproximación. Para el número exacto (por ejemplo, antes de enviar un documento largo) la API ofrece `responses.input_tokens.count`. La diferencia suele deberse a los tokens de formato que añade el propio mensaje.

## Sección 4: Parámetros de generación en modelos de razonamiento

Usaremos noticias tecnológicas para ver cómo los parámetros cambian el comportamiento.

**Celda 9: Efecto de `reasoning.effort`**: un problema que requiere pensar

In [ ]:
import time

problema = (
    "Un centro de datos tiene 3 racks. El rack A consume el doble que el B, y el C consume 4 kW menos que el A. "
    "Entre los tres consumen 46 kW. ¿Cuánto consume cada rack? Da solo los tres números."
)

for esfuerzo in ["none", "low", "medium"]:
    t0 = time.time()
    r = client.responses.create(model=MODELO, input=problema, reasoning={"effort": esfuerzo})
    seg = time.time() - t0
    razon = r.usage.output_tokens_details.reasoning_tokens
    print(f"effort={esfuerzo:<7} {seg:5.1f} s  {razon:5d} tokens de razonamiento  ->  {r.output_text.strip()}")

Solución correcta: A = 20 kW, B = 10 kW, C = 16 kW.

Más esfuerzo = más tokens de razonamiento = más latencia y coste, pero más fiabilidad en problemas con varios pasos. Regla práctica:
- `none`: clasificar, extraer, reformular (tareas «de reflejo»)
- `low`: el punto de partida razonable para casi todo (lo usamos por defecto)
- `medium` o más: matemáticas, código, planificación, análisis con varios pasos

**Celda 10: ¿Y `temperature`?**

In [ ]:
noticia = (
    "Apple ha anunciado su nuevo chip M4 Ultra con Neural Engine de 32 núcleos, "
    "capaz de procesar 40 billones de operaciones por segundo y ejecutar modelos "
    "de 70B parámetros directamente en el dispositivo, sin servidores externos."
)

# 1) temperature con razonamiento activado -> la API lo rechaza
try:
    client.responses.create(model=MODELO, input=f"Escribe un titular para: {noticia}",
                            reasoning={"effort": "low"}, temperature=1.3)
except Exception as e:
    print("Con razonamiento:", type(e).__name__, "-", str(e)[:200])

# 2) Sin razonamiento (effort='none') podemos explorar la creatividad
print()
for i in range(3):
    try:
        r = client.responses.create(model=MODELO, input=f"Escribe un titular impactante para: {noticia}",
                                    reasoning={"effort": "none"}, temperature=1.3)
        print(f"Intento {i+1}: {r.output_text.strip()}")
    except Exception as e:
        print("Sin razonamiento:", type(e).__name__, "-", str(e)[:200])
        break

En los modelos clásicos, `temperature` era el mando principal: 0 = determinista, >1 = creativo. En los modelos de razonamiento, OpenAI pide **quitar `temperature` y `top_p` cuando el razonamiento está activado**: el modelo ajusta su propio muestreo mientras piensa. Si necesitas variedad, pide varias alternativas en el prompt («dame 3 titulares distintos») o usa `effort="none"`.

> Si la segunda parte también da error, tu modelo no acepta `temperature` ni sin razonamiento. Es otra forma de ver que en GPT-6 el control se hace con `reasoning.effort` y el prompt.

**Celda 11: `max_output_tokens`: el límite incluye el razonamiento**

In [ ]:
def mostrar(r):
    u = r.usage
    print(f"  estado: {r.status}  |  salida: {u.output_tokens} tokens (razonamiento: {u.output_tokens_details.reasoning_tokens})")
    print(f"  texto: {r.output_text!r}")

print("=== 40 tokens con effort='medium' ===")
mostrar(client.responses.create(model=MODELO, input=f"Resume para Twitter: {noticia}",
                                reasoning={"effort": "medium"}, max_output_tokens=40))

print()
print("=== 40 tokens con effort='none' ===")
mostrar(client.responses.create(model=MODELO, input=f"Resume para Twitter: {noticia}",
                                reasoning={"effort": "none"}, max_output_tokens=40))

`max_output_tokens` limita **todos** los tokens de salida, **incluidos los de razonamiento**. Con un límite pequeño y razonamiento activado, el modelo puede gastarse el presupuesto pensando y devolver la respuesta vacía o cortada (`status='incomplete'`).

En producción: deja margen en `max_output_tokens` y controla la longitud de la respuesta con el prompt o con `verbosity` (siguiente celda).

**Celda 12: `verbosity`: controlar la longitud sin cortar**

In [ ]:
for v in ["low", "high"]:
    r = client.responses.create(model=MODELO, input=f"Explica esta noticia para un blog técnico: {noticia}",
                                reasoning={"effort": "low"}, text={"verbosity": v})
    print(f"=== verbosity={v} ({r.usage.output_tokens} tokens) ===")
    print(r.output_text)
    print()

`verbosity` le pide al modelo que sea más escueto o más detallado, sin el corte brusco de `max_output_tokens`.

## Sección 5: Conversación multi-turno

El modelo **no recuerda** por defecto: cada llamada es independiente. Hay dos formas de darle memoria.

**Celda 13: Opción A: enviar tú el historial completo**

In [ ]:
historial = [
    {"role": "system", "content": "Eres un asistente experto en tecnología e IA. Responde en 2-3 frases máximo."}
]

def chat(mensaje):
    historial.append({"role": "user", "content": mensaje})
    r = client.responses.create(model=MODELO, reasoning={"effort": ESFUERZO}, input=historial)
    historial.append({"role": "assistant", "content": r.output_text})
    return r.output_text

preguntas = [
    "¿Qué es un chip Neural Engine?",
    "¿Y cuántos parámetros pueden tener los modelos que ejecuta?",
    "¿Qué ventaja tiene ejecutarlos localmente frente a en la nube?",
]

for p in preguntas:
    print(f"Usuario:    {p}")
    print(f"Asistente:  {chat(p)}")
    print()

La **tercera pregunta** no dice de qué habla, y el modelo responde correctamente porque el historial completo se envía en cada llamada. Es lo que se hace con Chat Completions y con cualquier proveedor.

**Celda 14: Ver el historial que ve el modelo**

In [ ]:
print("=== HISTORIAL ENVIADO AL MODELO EN CADA LLAMADA ===")
for msg in historial:
    texto = msg["content"][:90] + "..." if len(msg["content"]) > 90 else msg["content"]
    print(f"[{msg['role'].upper()}]\n  {texto}\n")
print(f"Total: {len(historial)} mensajes enviados en cada petición")

El historial crece con cada turno: más tokens de entrada = más coste. En producción se usa una ventana deslizante (descartar los mensajes viejos) o se resume el historial.

**Celda 15: Opción B: memoria en el servidor con `previous_response_id`** (solo Responses API)

In [ ]:
instrucciones = "Eres un asistente experto en tecnología e IA. Responde en 2-3 frases máximo."

anterior = None
for p in preguntas:
    r = client.responses.create(
        model=MODELO,
        reasoning={"effort": ESFUERZO},
        instructions=instrucciones,        # las instructions NO se heredan: hay que reenviarlas
        input=p,                           # solo el mensaje nuevo
        previous_response_id=anterior,     # ...y la referencia a la respuesta anterior
    )
    anterior = r.id
    print(f"Usuario:    {p}")
    print(f"Asistente:  {r.output_text}")
    print(f"            (tokens de entrada facturados: {r.usage.input_tokens})\n")

Ahora solo enviamos el mensaje nuevo y el `id` de la respuesta anterior: OpenAI reconstruye la conversación en el servidor. El código es más simple, pero fíjate en los tokens de entrada: **el historial se sigue facturando**, aunque no lo envíes tú.

¿Cuándo usar cada opción? Historial manual si quieres control total (o cambiar de proveedor); `previous_response_id` para simplificar y para conservar el razonamiento entre turnos.

## Sección 6: Ingeniería de prompts

**Celda 16: System prompt: periodista tecnológica**

In [ ]:
r = client.responses.create(
    model=MODELO,
    reasoning={"effort": ESFUERZO},
    instructions=(
        "Eres Elena Vargas, periodista tecnológica con 15 años de experiencia. "
        "Escribes con estilo directo usando analogías para no técnicos. "
        "Siempre terminas con una reflexión sobre el impacto social. Máximo 3 párrafos."
    ),
    input="Explica qué es un transformer en IA y por qué fue revolucionario.",
)
print(r.output_text)

El **system prompt** (en Responses, `instructions`) define la persona, el estilo y las restricciones del modelo. En aplicaciones reales contiene instrucciones permanentes, restricciones de seguridad y contexto del dominio.

**Celda 17: Few-shot prompting: clasificador de noticias**

In [ ]:
def clasificar(titular):
    r = client.responses.create(
        model=MODELO,
        reasoning={"effort": "none"},   # clasificar es una tarea «de reflejo»: no hace falta razonar
        input=[
            {"role": "system",    "content": "Clasifica titulares en: tecnologia, politica, economia, ciencia, deportes. Solo la categoría."},
            {"role": "user",      "content": "Tesla presenta Optimus, su robot humanoide para fábricas"},
            {"role": "assistant", "content": "tecnologia"},
            {"role": "user",      "content": "El BCE sube los tipos al 4.5% para frenar la inflación"},
            {"role": "assistant", "content": "economia"},
            {"role": "user",      "content": "El Congreso aprueba la nueva ley de presupuestos"},
            {"role": "assistant", "content": "politica"},
            {"role": "user",      "content": "La ESA detecta agua líquida en la luna Europa de Júpiter"},
            {"role": "assistant", "content": "ciencia"},
            {"role": "user",      "content": titular},
        ],
    )
    return r.output_text.strip()

titulares_test = [
    "OpenAI lanza GPT-6 con cuatro modelos de distinto tamaño",
    "El Real Madrid gana la Champions por decimosexta vez",
    "La inflación en España baja al 2.1% en mayo",
    "Investigadores del CERN descubren una nueva partícula subatómica",
    "El gobierno aprueba el plan de pensiones para 2027",
    "Amazon despide al 8% de su plantilla en un recorte global",
]

print("Clasificación de titulares:")
for t in titulares_test:
    print(f"  [{clasificar(t).upper():12s}]  {t}")

El **few-shot prompting** guía al modelo con ejemplos sin reentrenarlo. El último titular es ambiguo (tecnología/economía): los casos límite revelan las limitaciones.

> En la Clase 2 veremos la **Decisions API**, pensada justo para esto: preguntas con respuestas cerradas, mucho más rápida y con la probabilidad de cada opción.

**Celda 18: Razonamiento: ¿el «piensa paso a paso» sigue haciendo falta?**

In [ ]:
titular = "Un estudiante de 14 años crea una IA que diagnostica cáncer con 99.9% de precisión usando selfies"

prompt_cot = (
    "Analiza si este titular es verosímil, exagerado o probablemente falso.\n"
    "Razona PASO A PASO antes del veredicto final:\n"
    "1. ¿Qué afirma exactamente?\n"
    "2. ¿Qué conocimiento técnico/científico se necesita para evaluarlo?\n"
    "3. ¿Hay inconsistencias o exageraciones?\n"
    "4. VEREDICTO: (verosímil / exagerado / probablemente falso) + 1 frase.\n\n"
    f'Titular: "{titular}"'
)
prompt_directo = (
    "¿Este titular es verosímil, exagerado o probablemente falso? Veredicto + 2 frases de justificación.\n\n"
    f'Titular: "{titular}"'
)

print("=== A) Chain-of-thought en el prompt, sin razonamiento interno (effort='none') ===")
print(client.responses.create(model=MODELO, input=prompt_cot, reasoning={"effort": "none"}).output_text)
print()
print("=== B) Prompt directo, con razonamiento interno (effort='medium') ===")
print(client.responses.create(model=MODELO, input=prompt_directo, reasoning={"effort": "medium"}).output_text)

El **chain-of-thought** («razona paso a paso») fue la técnica estrella con los modelos clásicos: obligar al modelo a escribir su razonamiento mejoraba mucho la precisión.

Los modelos de razonamiento hacen ese trabajo **internamente** (los tokens de razonamiento). Por eso:
- Con `effort="none"`, el CoT en el prompt sigue siendo útil.
- Con razonamiento activado, basta con pedir el resultado y el formato: añadir «piensa paso a paso» aporta poco y gasta tokens.
- Lo que sí sigue importando: dar contexto, definir el objetivo y especificar el formato de salida.